# 14.15 只練1、3、5輪，可以在2、4輪交卷嗎？


可以把推論程式設成二輪或四輪，但是否答得好要測。這個問題最有意思的地方，是**可執行**和**可泛化**並不相同。

要做乾淨的機制實驗，訓練時只抽1、3、5作終止深度，每次只在末端算代價；不要在中間所有輪都計分。訓練到五輪時，第二、四輪仍然是通往第五輪的內部狀態，也收到間接梯度，所以應稱它們為「**未直接監督的終止深度**」。不能說完全沒有見過第二、四輪的狀態。

凍結模型後，在相同的測試題上測1、2、3、4、5、6輪。二、四在訓練終止深度範圍內，叫深度**插值**；六超出最大終止深度，叫**外推**。這些詞標示測試位置，沒有承諾連續平滑。固定三輪的訓練也要另做一份對照，才能看出變動終止配方改了什麼。

下面是反例：用手工更新每輪把狀態變號，正確類別總是正號。它沒有學習，也不是 Transformer 效果，只證明「所有奇數輪好」在邏輯上推不出「偶數輪也好」。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
state = -1
for r in range(1, 7):
    state = -state
    probability_correct = 0.9 if state > 0 else 0.1
    print(r, probability_correct)

奇數輪答對的機率0.9，偶數輪0.1。共享更新可以產生這種週期，因此不能只憑共享參數或奇數輪訓練來保證偶數表現。實際網路未必這樣振盪，但要用完整曲線排除，而不是只挑表現最好的幾個點。

為了實際檢查，我們改用有限的「兩盒追蹤」任務。例如輸入「A=2；B=6；事件=交換；查詢=B」，正確答案是2：交換後A變6、B變2；若查詢A，正確答案則是6。資料也有把某盒設為指定值的事件；核心題包含1–4個事件，須按順序追到最後，只把預測整數與正確答案完全相同的題目算對。每一輪都處理同一份完整題目，並不是每輪另收到一個新事件；因此可以固定問題，比較不同交卷輪數。

本課實際從同一組隨機初始參數訓練兩份模型：固定3輪末端CE，與抽1／3／5輪末端CE。兩者的種子8159、每批128題、資料與1500次更新都相同，最後比較的是兩者實際第1500步的參數，不拿各自驗證最好的那一步混用。工作量以「一題經過一次共享區塊」為單位：固定3輪是 `1500次更新×128題×3輪=576,000` 次；抽1／3／5輪那份每次更新抽一個終止深度，同一批128題都跑那麼多輪；紀錄裡共580,096次，等於 `128×4532`，也就是1500次更新抽到的輪數加總是4532、平均每次約3.02輪，比固定3輪多 `580,096÷576,000−1≈0.71%`。這是相同更新次數的比較，並非相同算力。

以下是同一份兩盒追蹤測試資料的實測：共2956題，來自1478個家族。這裡的家族（[5.12](https://birdhackor.github.io/tiny-perceptron-vlm/5.12.html)）是同一組初始盒值與事件，分別問A、問B兩題，所以 `1478×2=2956`；切分時同一家族整組放在同一邊，測試題的初始值與事件組合都沒在訓練出現過。每格先列CE，再列答案正確率；越低的CE與越高的正確率分別回答不同問題。

| 推論輪數 | 固定3輪訓練：CE／正確率 | 抽1／3／5訓練：CE／正確率 |
| --- | --- | --- |
| 1 | 2.9575／39.45% | 1.1735／53.08% |
| 2 | 0.0931／97.56% | 0.1011／97.33% |
| 3 | 0.0238／99.49% | 0.0389／98.65% |
| 4 | 0.0285／98.99% | 0.0395／98.65% |
| 5 | 0.0618／98.31% | 0.0450／98.55% |
| 6 | 0.1072／97.67% | 0.0515／98.41% |

下圖由繪圖程式產生，標籤是英文。對照如下：最上方兩行標題 Same initialization and 1,500 updates 與 Heldout box tracking，是「同一初始化、各1500次更新」與「兩盒追蹤的留出題（這裡是測試資料）」；上圖縱軸 Cross-entropy (log scale; lower is better) 是「交叉熵（對數刻度，越低越好）」，下圖縱軸 Accuracy (%) 是「正確率（%）」，橫軸 Inference loops 是「推論輪數」；圖例 fixed terminal depth 3 是「固定3輪訓練」，sampled terminal depth 1/3/5 是「抽1／3／5輪訓練」。


In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/looped-mechanism-measured.svg")))

藍線是固定3輪，橘線是抽1／3／5輪。上圖CE越低越好，縱軸用對數刻度，避免第一輪的大值把後面的差異壓扁；下圖正確率越高越好。沿橘線看2、4輪：本次確實能在未直接監督的終止深度交卷；看第6輪則是在已訓練終止深度外的表現。這是一個種子和有限任務的結果，不是偶數輪或外推必然成功的定理；固定3輪模型在第3輪的成績也更高。

這兩份是機制對照。[19.15](https://birdhackor.github.io/tiny-perceptron-vlm/19.15.html)的 Looped 解題器另用種子7421，直接監督1–5輪；它不是本表的同起點對照。表中的CE與正確率記在[實驗紀錄](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/lesson-products/results/final-report-v1.json)的 `mechanisms`：`fixed3` 是固定3輪、`odd` 是抽1／3／5輪，`evaluation` 下依推論輪數列出 `boxes` 的 `cross_entropy`、`accuracy`、`count` 與 `families`。種子、每批題數、更新次數與工作量記在 [mechanism-fixed3-runner-result.json](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/lesson-products/results/mechanism-fixed3-runner-result.json) 與 [mechanism-odd135-runner-result.json](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/lesson-products/results/mechanism-odd135-runner-result.json) 的 `execution.training`，工作量看其中的 `work.block_sequence_work`。

練習：如果加入第2輪監督而仍不監督第4輪，該如何標示？要完整寫監督集合與抽樣規則，第2輪不再是未直接監督；不要只用「跳躍式訓練」一個名稱掩蓋差異。
